# Frequency Guard - Deploy from GitHub to Colab
Deploy your Frequency Guard project from GitHub with ngrok public link

## Step 1: Install Node.js

In [ ]:
%%bash
# Install Node.js 20.x
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
sudo apt-get install -y nodejs
echo "Node version: $(node --version)"
echo "NPM version: $(npm --version)"

## Step 2: Clone Project from GitHub

In [ ]:
# ⚙️ Configure your GitHub repository
GITHUB_USERNAME = "YOUR_USERNAME"  # Replace with your GitHub username
REPO_NAME = "frequency-guard"      # Replace with your repo name

# Clone repository
!rm -rf /content/{REPO_NAME}
!git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git /content/{REPO_NAME}

print("\n✅ Repository cloned successfully!")
print(f"📁 Project location: /content/{REPO_NAME}")
!ls -la /content/{REPO_NAME}

## Step 3: Install Project Dependencies

In [ ]:
import os
import subprocess

print("📦 Installing dependencies...")
print("-" * 70)

# Change to project directory
os.chdir(f'/content/{REPO_NAME}')

# Run npm install
result = subprocess.run(
    ['npm', 'install'],
    capture_output=True,
    text=True
)

# Show output
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)

# Check if installation was successful
if result.returncode != 0:
    print("\n❌ ERROR: npm install failed!")
    print(f"Exit code: {result.returncode}")
    raise Exception("Dependency installation failed")

print("\n" + "="*70)
print("✅ Dependencies installed successfully!")
print("="*70)

# Verify Vite is installed
vite_path = '/content/{}/node_modules/.bin/vite'.format(REPO_NAME)
if os.path.exists(vite_path):
    print(f"✅ Vite found at: {vite_path}")
else:
    print(f"⚠️  Warning: Vite binary not found at {vite_path}")
    print("This may cause issues in Step 5")

# Show key installed packages
print("\n📋 Verifying key packages:")
try:
    pkg_result = subprocess.run(
        ['npm', 'list', 'vite', 'react', 'react-dom', '--depth=0'],
        capture_output=True,
        text=True
    )
    print(pkg_result.stdout)
except:
    pass

## Step 4: Setup ngrok

In [ ]:
%%bash
# Download and install ngrok
echo "📥 Downloading ngrok..."
wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
tar -xzf ngrok-v3-stable-linux-amd64.tgz
chmod +x ngrok
sudo mv ngrok /usr/local/bin/
echo "✅ ngrok installed: $(ngrok version)"

In [ ]:
# 🔑 Add your ngrok authtoken
# Get your free token from: https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"

!ngrok authtoken {NGROK_AUTH_TOKEN}
print("✅ ngrok configured successfully!")

## Step 5: Deploy and Get Public Link 🚀

In [ ]:
import subprocess
import time
import requests
import socket
import threading
import os
from IPython.display import display, HTML, clear_output

def check_port(port, max_attempts=30):
    """Check if a port is open and accepting connections"""
    for i in range(max_attempts):
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.settimeout(1)
            result = sock.connect_ex(('127.0.0.1', port))
            sock.close()
            if result == 0:
                return True
        except:
            pass
        time.sleep(1)
    return False

def monitor_output(process, name):
    """Monitor and print process output in real-time"""
    for line in iter(process.stdout.readline, b''):
        if line:
            print(f"[{name}] {line.decode('utf-8').strip()}")

print("🚀 Starting Frequency Guard...\n")

# Verify Vite is installed
vite_bin = f'/content/{REPO_NAME}/node_modules/.bin/vite'
if not os.path.exists(vite_bin):
    print("❌ ERROR: Vite is not installed!")
    print(f"Expected location: {vite_bin}")
    print("\n🔧 Solution: Go back and run Step 3 (Install Dependencies)")
    print("Make sure you see '✅ Dependencies installed successfully!'")
    raise Exception("Vite not found - please run Step 3")

print("✅ Vite found, starting server...")

# Start Vite dev server using npx to ensure correct binary
print("⚡ Starting Vite development server...")
vite_process = subprocess.Popen(
    ["npx", "vite", "--host", "0.0.0.0", "--port", "5173"],
    cwd=f"/content/{REPO_NAME}",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1
)

# Start monitoring Vite output in a thread
vite_thread = threading.Thread(target=monitor_output, args=(vite_process, "Vite"), daemon=True)
vite_thread.start()

# Wait for Vite server to be ready
print("⏳ Waiting for server to start (checking port 5173)...")
if not check_port(5173, max_attempts=30):
    print("\n❌ ERROR: Vite server failed to start!")
    print("📋 Checking process status...")
    
    # Check if process is still running
    if vite_process.poll() is not None:
        print(f"⚠️  Process exited with code: {vite_process.poll()}")
    
    print("\n🔧 Common issues:")
    print("1. Dependencies not installed correctly - rerun Step 3")
    print("2. Port 5173 already in use - restart runtime")
    print("3. Node.js version issue - restart from Step 1")
    raise Exception("Server startup failed")

print("✅ Vite server is running on port 5173\n")

# Start ngrok tunnel
print("🌐 Creating ngrok tunnel...")
ngrok_process = subprocess.Popen(
    ["ngrok", "http", "5173", "--log=stdout"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# Wait for ngrok to start
print("⏳ Waiting for ngrok to initialize...")
time.sleep(5)

# Get ngrok public URL
try:
    response = requests.get("http://localhost:4040/api/tunnels", timeout=10)
    tunnels = response.json()["tunnels"]
    
    if not tunnels:
        raise Exception("No tunnels found")
    
    public_url = tunnels[0]["public_url"]
    
    clear_output(wait=True)
    
    print("\n" + "="*70)
    print("🎉 FREQUENCY GUARD IS LIVE!")
    print("="*70)
    print(f"\n🔗 Public URL: {public_url}")
    print("\n📱 Share this link with anyone to access your app")
    print("⏰ Keep this cell running to maintain the connection")
    print("🛑 Click 'Stop' button to shut down the server\n")
    print("="*70 + "\n")
    
    # Display clickable link
    display(HTML(f'''
        <div style="text-align: center; padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 10px; margin: 20px 0;">
            <h1 style="color: white; margin-bottom: 20px;">🎊 Deployment Successful!</h1>
            <a href="{public_url}" target="_blank"
               style="display: inline-block; padding: 15px 40px; background: white; color: #667eea;
                      text-decoration: none; border-radius: 5px; font-size: 20px; font-weight: bold;
                      box-shadow: 0 4px 6px rgba(0,0,0,0.2); transition: transform 0.2s;"
               onmouseover="this.style.transform='scale(1.05)'"
               onmouseout="this.style.transform='scale(1)'">
                🚀 Open Frequency Guard
            </a>
            <p style="color: white; margin-top: 20px; font-size: 14px;">{public_url}</p>
        </div>
    '''))
    
    print("\n✨ Server is running...")
    print("📊 View logs at: http://localhost:4040")
    print("-" * 70 + "\n")
    
    # Keep running and show status
    try:
        while vite_process.poll() is None:
            time.sleep(1)
        print("\n⚠️  Vite server stopped unexpectedly")
    except KeyboardInterrupt:
        print("\n🛑 Shutting down...")
    finally:
        vite_process.terminate()
        ngrok_process.terminate()
        print("✅ Shutdown complete")
        
except requests.exceptions.RequestException as e:
    print(f"\n❌ Error connecting to ngrok API: {e}")
    print("\n🔧 Troubleshooting:")
    print("1. Check if ngrok authtoken is configured correctly")
    print("2. Visit ngrok dashboard: https://dashboard.ngrok.com/")
    print("3. Try manually: http://localhost:4040")
    
    # Clean up
    vite_process.terminate()
    ngrok_process.terminate()
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    print("\n🔧 Troubleshooting:")
    print("1. Restart runtime and run all cells again")
    print("2. Check ngrok token is valid")
    print("3. Verify repository was cloned correctly")
    
    # Clean up
    vite_process.terminate()
    ngrok_process.terminate()

## 📝 Important Notes

### ✅ What to do:
- **Keep Step 5 cell running** to maintain the live connection
- Share the public URL with anyone who needs access
- Access ngrok inspector at: http://localhost:4040 for request logs

### ⚠️ Limitations:
- **Free ngrok tunnels expire after 2 hours** - restart Step 5 to get a new URL
- Colab runtime disconnects after ~12 hours of inactivity
- Don't close this browser tab while the server is running

### 🔧 Troubleshooting:
If something goes wrong:
1. Restart runtime: `Runtime > Restart runtime`
2. Run all cells from Step 1 again
3. Check GitHub repo is public or you have access
4. Verify ngrok token at: https://dashboard.ngrok.com/